# (01) Import and settings

In this section we import all required libraries, configure global settings and fix the random seed for reproducibility.


In [ ]:
# (01) Import and settings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Global display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Plotting style
sns.set(style="whitegrid")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# (02) Loading dataset

We load `titanic.csv` from the local directory, inspect the basic structure and confirm that the target column is `Survived`.  
All remaining columns are considered candidate input features for the classification task.


In [ ]:
# (02) Loading dataset

# Load Titanic dataset
data = pd.read_csv("titanic.csv")

# Basic info
print("Shape:", data.shape)
print("\nInfo:")
print(data.info())

print("\nFirst rows:")
display(data.head())

# Separate features and target
TARGET_COL = "Survived"
assert TARGET_COL in data.columns, "Target column 'Survived' not found in titanic.csv"

X = data.drop(columns=[TARGET_COL])
y = data[TARGET_COL].astype(int)

print("\nTarget distribution:")
print(y.value_counts(normalize=True).rename("ratio").to_frame())


: 

# (03) Explorative Data Analysis

We perform a compact EDA to understand feature distributions, missing values and basic relationships to survival, for example:
- Overall missing value pattern
- Survival rate by sex and passenger class
- Age and fare distributions

This helps guide sensible feature engineering and model design.


In [ ]:
# (03) Explorative Data Analysis

# Basic statistics for numeric and categorical features
print("Numeric summary:")
display(X.describe())

print("\nCategorical summary (top categories):")
display(X.describe(include="object"))

# Missing values overview
print("\nMissing values per column:")
missing = X.isna().sum().sort_values(ascending=False)
display(missing.to_frame(name="missing_count"))

# A few key plots (only if columns exist)

def safe_countplot(x, hue=None, data=None, title=None):
    if x in data.columns:
        plt.figure(figsize=(6, 4))
        sns.countplot(data=data, x=x, hue=hue)
        plt.title(title or f"{x} distribution")
        plt.tight_layout()
        plt.show()

# Combine X and y for plotting
df_plot = X.copy()
df_plot[TARGET_COL] = y

# Survival by Sex
safe_countplot(
    x="Sex", 
    hue=TARGET_COL, 
    data=df_plot, 
    title="Survival by Sex"
)

# Survival by Pclass
safe_countplot(
    x="Pclass", 
    hue=TARGET_COL, 
    data=df_plot, 
    title="Survival by Passenger Class"
)

# Age distribution by survival (if Age exists)
if "Age" in df_plot.columns:
    plt.figure(figsize=(6, 4))
    sns.kdeplot(
        data=df_plot,
        x="Age",
        hue=TARGET_COL,
        common_norm=False,
        fill=True,
        alpha=0.4
    )
    plt.title("Age distribution by survival")
    plt.tight_layout()
    plt.show()

# Fare distribution (log scaled) if Fare exists
if "Fare" in df_plot.columns:
    plt.figure(figsize=(6, 4))
    sns.histplot(df_plot["Fare"], bins=40)
    plt.title("Fare distribution")
    plt.tight_layout()
    plt.show()


# (04) Feature Engineering (Imputation, new features, encoding, scaling, cleanup)

Goal: build a modular, reusable preprocessing and feature engineering pipeline that produces model ready numeric matrices for both KNN and DecisionTreeClassifier.

Key design decisions:
- Work with at least 12 meaningful features based on typical Titanic columns.
- Create robust domain features:
  - `FamilySize` and `IsAlone` from SibSp and Parch
  - Passenger `Title` extracted from Name
  - `CabinKnown` flag from Cabin availability
  - `TicketGroupSize` from ticket counts
  - `IsChild` based on age
  - `TicketPrefix` from ticket text
- Handle missing values with `SimpleImputer` (median for numeric, most frequent for categorical).
- Encode categorical variables with `OneHotEncoder` (handle unknown categories).
- Scale numeric features with `StandardScaler`, useful for KNN while still acceptable for tree models.
- Implement everything as a custom transformer plus `ColumnTransformer` and `Pipeline`, so the model can operate directly on raw rows.


In [ ]:
# (04) Feature Engineering

class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom transformer that:
    - Creates domain features for the Titanic dataset
    - Drops unused or redundant columns
    - Returns a pandas DataFrame for downstream ColumnTransformer
    """

    def __init__(self):
        # Define columns we expect and how we will treat them
        self.base_cols_to_keep_ = None

    def fit(self, X, y=None):
        # No learned parameters here, but we can remember which columns existed
        self.base_cols_to_keep_ = list(X.columns)
        return self

    def transform(self, X):
        # Work on a copy to avoid side effects
        X = X.copy()

        # Ensure we are working with a DataFrame
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.base_cols_to_keep_)

        # Helper: safely get a column or a default
        def safe_col(col, default=np.nan):
            return X[col] if col in X.columns else pd.Series(default, index=X.index)

        # Title extraction from Name
        if "Name" in X.columns:
            titles = X["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
            # Map rare titles into a few common groups
            title_map = {
                "Mlle": "Miss",
                "Ms": "Miss",
                "Mme": "Mrs",
                "Lady": "Royalty",
                "Countess": "Royalty",
                "Sir": "Royalty",
                "Don": "Royalty",
                "Dona": "Royalty",
                "Jonkheer": "Royalty",
                "Col": "Officer",
                "Major": "Officer",
                "Capt": "Officer",
                "Dr": "Officer",
                "Rev": "Officer"
            }
            titles = titles.replace(title_map)
            X["Title"] = titles.fillna("Unknown")
        else:
            X["Title"] = "Unknown"

        # Family size from SibSp + Parch
        sibsp = safe_col("SibSp", 0).fillna(0)
        parch = safe_col("Parch", 0).fillna(0)
        X["FamilySize"] = sibsp + parch + 1

        # IsAlone flag
        X["IsAlone"] = (X["FamilySize"] == 1).astype(int)

        # CabinKnown flag
        if "Cabin" in X.columns:
            X["CabinKnown"] = X["Cabin"].notna().astype(int)
        else:
            X["CabinKnown"] = 0

        # Ticket group size: number of passengers with the same ticket
        if "Ticket" in X.columns:
            ticket_counts = X["Ticket"].value_counts()
            X["TicketGroupSize"] = X["Ticket"].map(ticket_counts).fillna(1)
        else:
            X["TicketGroupSize"] = 1

        # Ticket prefix (first token of ticket that contains letters)
        if "Ticket" in X.columns:
            ticket_prefix = (
                X["Ticket"]
                .astype(str)
                .str.replace(r"[./]", " ", regex=True)
                .str.split()
                .apply(lambda parts: next((p for p in parts if not p.isdigit()), "None"))
            )
            X["TicketPrefix"] = ticket_prefix
        else:
            X["TicketPrefix"] = "None"

        # IsChild (for example age < 16)
        age_series = safe_col("Age")
        X["IsChild"] = (age_series < 16).astype("float")
        # Missing ages will become False (0), imputed later as numeric

        # Ensure Sex and Embarked exist for downstream encoding
        if "Sex" not in X.columns:
            X["Sex"] = "Unknown"
        if "Embarked" not in X.columns:
            X["Embarked"] = "Unknown"

        if "Pclass" not in X.columns:
            # If missing, fall back to 3 as default
            X["Pclass"] = 3
        if "Fare" not in X.columns:
            X["Fare"] = 0.0

        # Drop columns that should not be passed to the model directly
        cols_to_drop = [c for c in ["PassengerId", "Name", "Cabin", "Ticket"] if c in X.columns]
        X = X.drop(columns=cols_to_drop)

        return X


# Define final feature lists after feature engineering
numeric_features = [
    "Pclass",
    "Age",
    "Fare",
    "FamilySize",
    "TicketGroupSize"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title",
    "CabinKnown",
    "IsAlone",
    "IsChild",
    "TicketPrefix"
]

print("Planned numeric features:", numeric_features)
print("Planned categorical features:", categorical_features)

# Define preprocessing for numeric and categorical features
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# ColumnTransformer that applies the above transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

print("\nPreprocessor and feature engineering are configured.")


# (05) Train test split

We split the data into training and test sets before any model fitting or hyperparameter tuning:

- Training set: used for cross validation, model selection and hyperparameter tuning.
- Test set: kept completely separate and used once at the end for unbiased evaluation.

Stratification on `Survived` preserves the class balance in both splits.


In [ ]:
# (05) Train test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).rename("ratio").to_frame())
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).rename("ratio").to_frame())


# (06) ML pipeline (KNN and Decision Tree)

We now build two complete pipelines:

1. KNN classifier pipeline
2. DecisionTreeClassifier pipeline

Both pipelines:
- Start with the custom `TitanicFeatureEngineer`
- Apply the shared `preprocessor` (`ColumnTransformer`)
- Finish with the model step (`KNeighborsClassifier` or `DecisionTreeClassifier`)

This avoids duplicated logic and ensures that raw raw rows can be passed directly to each pipeline.


In [ ]:
# (06) ML pipeline (KNN and Decision Tree)

# Base pipeline that encapsulates feature engineering and preprocessing
def make_base_pipeline(model):
    return Pipeline(
        steps=[
            ("feature_engineering", TitanicFeatureEngineer()),
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

knn_pipeline = make_base_pipeline(
    KNeighborsClassifier()
)

dt_pipeline = make_base_pipeline(
    DecisionTreeClassifier(random_state=RANDOM_STATE)
)

print(knn_pipeline)
print("\n")
print(dt_pipeline)


# (07) Cross validation and fit of base models

We evaluate both base pipelines using stratified k fold cross validation on the training set only.

Steps:
- Use `StratifiedKFold` with shuffling for consistent splits.
- Compute cross validated accuracy for both models with `cross_val_score`.
- Inspect the distribution, mean and standard deviation of scores.
- After that, fit both pipelines on the full training set (without test data).


In [ ]:
# (07) Cross validation and fit of base models

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

def evaluate_cv(pipeline, X, y, cv, name="model"):
    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )
    print(f"{name} CV accuracy scores:", scores)
    print(f"{name} CV mean accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
    return scores

print("Base KNN:")
knn_cv_scores = evaluate_cv(knn_pipeline, X_train, y_train, cv, name="KNN")

print("\nBase Decision Tree:")
dt_cv_scores = evaluate_cv(dt_pipeline, X_train, y_train, cv, name="DecisionTree")

# Visual comparison of CV scores
plt.figure(figsize=(6, 4))
cv_results_df = pd.DataFrame(
    {
        "KNN": knn_cv_scores,
        "DecisionTree": dt_cv_scores
    }
)
sns.boxplot(data=cv_results_df)
plt.title("Cross validation accuracy comparison (base models)")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.show()

# Fit both base models on the full training set
knn_pipeline.fit(X_train, y_train)
dt_pipeline.fit(X_train, y_train)


# (08) Hyperparameter tuning

We tune both models independently using grid search with cross validation on the training data.

Design choices:
- KNN:
  - `n_neighbors`: small odd values in a reasonable range
  - `weights`: uniform vs distance based
  - `p`: 1 or 2 for Manhattan vs Euclidean distances
- DecisionTreeClassifier:
  - `max_depth`: control tree depth
  - `min_samples_split` and `min_samples_leaf`: regularization
  - `criterion`: gini, entropy or log loss

We use `GridSearchCV` with the same stratified k fold strategy to balance thoroughness and runtime.


In [ ]:
# (08) Hyperparameter tuning

# KNN hyperparameter grid
knn_param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

knn_grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0
)

print("Running GridSearchCV for KNN...")
knn_grid_search.fit(X_train, y_train)
print("Best KNN params:", knn_grid_search.best_params_)
print("Best KNN CV accuracy:", knn_grid_search.best_score_)

# Decision Tree hyperparameter grid
dt_param_grid = {
    "model__max_depth": [None, 3, 5, 7, 9],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__criterion": ["gini", "entropy", "log_loss"],
}

dt_grid_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0
)

print("\nRunning GridSearchCV for Decision Tree...")
dt_grid_search.fit(X_train, y_train)
print("Best Decision Tree params:", dt_grid_search.best_params_)
print("Best Decision Tree CV accuracy:", dt_grid_search.best_score_)

# Extract best estimators
best_knn = knn_grid_search.best_estimator_
best_dt = dt_grid_search.best_estimator_


# (09) Evaluation

We now evaluate the tuned best estimators on the held out test set.

For each model we compute:
- Accuracy
- Precision
- Recall
- F1 score
- Confusion matrix (numbers and heatmap)
- ROC AUC (if probabilities are available)

This gives a detailed view of how each model behaves in terms of correct predictions, false alarms and missed survivors.


In [ ]:
# (09) Evaluation

def evaluate_on_test(model, X_test, y_test, model_name="model"):
    y_pred = model.predict(X_test)

    metrics = {}
    metrics["accuracy"] = accuracy_score(y_test, y_pred)
    metrics["precision"] = precision_score(y_test, y_pred, zero_division=0)
    metrics["recall"] = recall_score(y_test, y_pred, zero_division=0)
    metrics["f1"] = f1_score(y_test, y_pred, zero_division=0)

    # ROC AUC if probability scores are available
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_test, y_proba)
    except Exception:
        metrics["roc_auc"] = np.nan

    print(f"\n=== {model_name} classification report ===")
    print(classification_report(y_test, y_pred, digits=3))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0 (died)", "Actual 1 (survived)"],
        columns=["Predicted 0", "Predicted 1"],
    )
    print(f"{model_name} confusion matrix:")
    display(cm_df)

    plt.figure(figsize=(4, 3))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{model_name} confusion matrix")
    plt.tight_layout()
    plt.show()

    return metrics

knn_test_metrics = evaluate_on_test(best_knn, X_test, y_test, model_name="Tuned KNN")
dt_test_metrics = evaluate_on_test(best_dt, X_test, y_test, model_name="Tuned Decision Tree")

print("\nKNN test metrics:", knn_test_metrics)
print("Decision Tree test metrics:", dt_test_metrics)


: 

# (10) Selection of best model and conclusion

We compare the tuned models using the test set metrics and select the primary model based on accuracy and F1, while also looking at precision, recall and ROC AUC.  
Finally, we summarize the full supervised learning workflow on the Titanic dataset: data, preprocessing, modeling, validation, tuning and selection.


In [ ]:
# (10) Selection of best model and conclusion

# Combine metrics into a DataFrame for easier comparison
comparison_df = pd.DataFrame(
    {
        "Tuned KNN": knn_test_metrics,
        "Tuned Decision Tree": dt_test_metrics,
    }
)

comparison_df


In [ ]:
# Determine best model based on primary metric (for example F1 score, then accuracy as tie breaker)

primary_metric = "f1"
secondary_metric = "accuracy"

best_model_name = comparison_df.loc[primary_metric].idxmax()
other_model_name = [m for m in comparison_df.columns if m != best_model_name][0]

print(f"Primary selection metric: {primary_metric}")
print(f"Best model on test set: {best_model_name}")

print("\nDetailed comparison:")
display(comparison_df.style.format("{:.3f}"))

print("\nConclusion:")
print(
    f"- The {best_model_name} achieved higher {primary_metric} on the held out test set compared to {other_model_name}.\n"
    f"- Looking at accuracy, precision, recall and ROC AUC, the selected model offers a good balance between correctly predicting survivors and avoiding false positives.\n"
    f"- The pipeline integrates feature engineering, robust missing value handling, encoding and scaling in a single reusable structure.\n"
    f"- K fold cross validation and grid search on the training set helped find sensible hyperparameters while keeping the test set untouched until the final evaluation.\n"
    f"- This setup is a solid template for supervised classification on structured tabular data beyond the Titanic example."
)
